# SASRec. Квантование с обучением (QAT) и оценка на CPU


За основу возьмем модель из оригинального репозитория
https://github.com/pmixer/SASRec.pytorch/blob/main/python/model.py   
Данные для обучения возьмем из того же репозитория
https://github.com/pmixer/SASRec.pytorch/blob/main/python/data/ml-1m.txt

In [1]:
# !mkdir data
# !wget https://raw.githubusercontent.com/pmixer/SASRec.pytorch/refs/heads/main/python/data/ml-1m.txt
# !mv ml-1m.txt data/.

In [2]:
import torch

import numpy as np
from torch import nn
from pathlib import Path
from typing import Any, Dict
from train import train_fp32, train_qat, apply_adaround

from models.quantization import QuantSASRec
from data.dataloader import create_dataloaders
from utils import load_config, set_random_seeds, ensure_dir

In [3]:
class Args:
    """Helper to pass config parameters to SASRec model."""
    def __init__(self, config: Dict[str, Any]):
        self.hidden_units = config["model"]["hidden_units"]
        self.num_blocks = config["model"]["num_blocks"]
        self.num_heads = config["model"]["num_heads"]
        self.dropout_rate = config["model"]["dropout_rate"]
        self.maxlen = config["model"]["maxlen"]
        self.device = torch.device(config["experiment"].get("device", "cuda" if torch.cuda.is_available() else "cpu"))
        self.norm_first = config["model"].get("norm_first", False)

In [4]:
config = load_config("/Users/admin/Desktop/work/ysda/QAT/EFF_DL_QAT/SasRec/configs/fp32.yaml")
set_random_seeds(config["experiment"].get("seed", 42))
args = Args(config)

In [5]:
train_loader, val_loader, test_loader, dataset = create_dataloaders(
        config, seed=config["experiment"].get("seed", 42))
[user_train, user_valid, user_test, usernum, itemnum] = dataset

Loading data from ./data/ml-1m.txt...


/Users/admin/Desktop/work/ysda/QAT/EFF_DL_QAT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 14 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [6]:
model = QuantSASRec(usernum, itemnum, args).to(args.device)
    
quant_cfg = config.get("quantization", {})
strategy_name = quant_cfg.get("method", "fp32").lower()
    
checkpoint_dir = Path(config.get("paths", {}).get("checkpoints_dir", "./checkpoints"))
run_name = config["experiment"].get("run_name", "sasrec_run")
checkpoint_subdir = checkpoint_dir / run_name
ensure_dir(checkpoint_subdir)
    
results_dir = Path(config.get("paths", {}).get("results_dir", "./results"))
ensure_dir(results_dir)
    
logging_cfg = config.get("logging", {})
logger = None
    
criterion = nn.BCEWithLogitsLoss()

In [7]:
train_fp32(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_fp32.pth",
            logger=logger
        )


Starting FP32 training...
FP32 training done. Best NDCG (Val): 0.0000
Loading best FP32 checkpoint from checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth for final testing...
Running final test evaluation (user_test split)...


[FP32][Test] NDCG@10: 0.0092 | Hit@10: 0.0204


In [10]:
train_fp32(
            model=model,
            train_loader=train_loader,
            dataset=dataset,
            config=config,
            criterion=criterion,
            args=args,
            checkpoint_dir=str(checkpoint_subdir),
            save_name="sasrec_fp32.pth",
            logger=logger
        )


Starting FP32 training...


Training:   0%|          | 0/48 [00:00<?, ?it/s]/Users/admin/Desktop/work/ysda/QAT/EFF_DL_QAT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 14 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/Users/admin/Desktop/work/ysda/QAT/EFF_DL_QAT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


[FP32] Epoch 1/20 | Loss: 6.2839 | NDCG@10: 0.0011 | Hit@10: 0.0030
Saved best FP32 checkpoint (NDCG: 0.0011)


[FP32] Epoch 2/20 | Loss: 5.4060 | NDCG@10: 0.0006 | Hit@10: 0.0015


[FP32] Epoch 3/20 | Loss: 3.8088 | NDCG@10: 0.0009 | Hit@10: 0.0022


...

[FP32] Epoch 17/20 | Loss: 0.9434 | NDCG@10: 0.0101 | Hit@10: 0.0215


[FP32] Epoch 18/20 | Loss: 0.9452 | NDCG@10: 0.0105 | Hit@10: 0.0222


[FP32] Epoch 19/20 | Loss: 0.9416 | NDCG@10: 0.0109 | Hit@10: 0.0238


[FP32] Epoch 20/20 | Loss: 0.9397 | NDCG@10: 0.0111 | Hit@10: 0.0233
FP32 training done. Best NDCG (Val): 0.0118
Loading best FP32 checkpoint from checkpoints/sasrec_runs/sasrec_fp32/sasrec_fp32.pth for final testing...


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 